> ## SOLUTION / ANSWER KEY &mdash; Lab 10.9
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-09-responsible-checklist.ipynb`](../lab-09-responsible-checklist.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.9 &mdash; The Responsible-Agent Checklist

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Turn the responsible-agent checklist into code
- Check grounding, least-privilege, HITL, observability, evals
- Gate deployment on every item passing

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; guardrail logic, tool wiring, agent structure, and reading a fixed real message trace &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; The responsible-agent checklist](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-10-09"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Before an agent ships, you should answer **yes** to every item on the responsible-agent checklist (deck
slide 11): grounded, least-privilege, human-in-the-loop, observable, evaluated. Encode it as **automated
checks** over the agent's config and make deployment a **gate** &mdash; no item skipped, no exceptions.

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "wire_funds"}
print("a config to gate:", {"grounds_and_cites": True, "tools": ["extract"], "human_approval": True, "traced": True, "eval_cases": 12})

## Your Turn
Complete `checklist` (per-item pass) and `ready_to_deploy` (all must pass).

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "wire_funds"}

def checklist(cfg):
    return {
        "grounded": cfg.get("grounds_and_cites", False),
        "least_privilege": not any(t in DANGEROUS for t in cfg.get("tools", [])),
        "human_in_loop": cfg.get("human_approval", False),
        "observable": cfg.get("traced", False),
        "evaluated": cfg.get("eval_cases", 0) > 0,
    }

def ready_to_deploy(cfg):
    return all(checklist(cfg).values())

GOOD = {'grounds_and_cites': True, 'tools': ['extract', 'compute'], 'human_approval': True, 'traced': True, 'eval_cases': 12}
try:
    print('good config:', ready_to_deploy(GOOD))
    print('checklist  :', checklist(GOOD))
    print('dangerous  :', ready_to_deploy({**GOOD, 'tools': ['place_trade']}))
    print('no evals   :', ready_to_deploy({**GOOD, 'eval_cases': 0}))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a compliant config passes every item", lambda: all(checklist(GOOD).values()) is True)
expect_true("a dangerous tool fails least_privilege", lambda: checklist({**GOOD, "tools": ["place_trade"]})["least_privilege"] is False)
expect_true("no human approval fails human_in_loop", lambda: checklist({**GOOD, "human_approval": False})["human_in_loop"] is False)
expect_true("no evals fails evaluated", lambda: checklist({**GOOD, "eval_cases": 0})["evaluated"] is False)
expect_true("ready_to_deploy is True only when all items pass", lambda: ready_to_deploy(GOOD) is True and ready_to_deploy({**GOOD, "traced": False}) is False)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
The checklist as a deployment gate makes responsibility non-optional: no agent ships unless it's grounded, least-privilege, human-gated, observable and evaluated. Governance you can run in CI, not a document nobody reads.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>